In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.pipeline import Pipeline

from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, mean_absolute_error

import joblib
import matplotlib.pyplot as plt

In [2]:
np.random.seed(42)
n= 1500

In [3]:
area_sqft = np.random.normal(loc=900, scale=300, size=n).clip(300, 3000)
distance_km_to_metro = np.random.exponential(scale=3.0, size=n).clip(0.2, 20)
num_rooms = np.random.choice([1, 2, 3, 4], size=n, p=[0.25, 0.45, 0.25, 0.05])
building_age_years = np.random.normal(loc=12, scale=8, size=n).clip(0, 50)
noise = np.random.normal(0, 2500, size=n)
rent_inr = (
    25 * area_sqft
    - 1800 * distance_km_to_metro
    + 8000 * num_rooms
    - 120 * building_age_years
    + 9000 * np.exp(-distance_km_to_metro)   # nonlinear premium near metro
    + noise
).clip(8000, 200000)

In [4]:
df = pd.DataFrame({
    "area_sqft": area_sqft,
    "distance_km_to_metro": distance_km_to_metro,
    "num_rooms": num_rooms,
    "building_age_years": building_age_years,
    "rent_inr": rent_inr
})

In [5]:
df

,area_sqft,distance_km_to_metro,num_rooms,building_age_years,rent_inr
0,1049.014246,0.943396,1,13.144390,31126.887661
1,858.520710,5.229755,2,18.796896,20397.105313
2,1094.306561,2.068994,3,17.542861,44916.610745
3,1356.908957,0.240568,3,16.660571,62970.313908
4,829.753988,0.200000,2,5.600476,43337.515799
...,...,...,...,...,...
1495,1501.827866,14.026507,2,25.476199,24768.312426
1496,1518.451073,4.240752,4,18.738379,60659.625682
1497,1262.509869,1.751769,2,12.185568,42471.325417
1498,1207.218758,3.686816,2,22.808126,33342.034202


In [6]:
x = df.drop("rent_inr", axis=1)
y = df["rent_inr"]

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [24]:
base_model = Pipeline (
steps=[
    ('svr', SVR(kernel='linear', C=100, epsilon=0.1))
]
)

In [25]:
base_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['area_sqft','distance_km_to_metro','num_rooms','building_age_years']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",100
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'


In [26]:
base_model.predict(x_test)


array([48124.88878319, 43584.08122751, 33375.45590254, 43334.35731344,
       41524.79870473, 14835.43020929, 12058.38033934, 37944.27384446,
       35813.41936826, 34741.92369418, 49933.77605763, 45402.4660411 ,
       40385.88582923, 51552.32318846, 53328.70707022, 27063.41356907,
       45628.35809359, 23225.69683664, 29231.80913877, 26043.80413844,
       58725.33105759, 26491.43189315, 33229.99600859, 23048.05817663,
       37073.66775284, 35729.59262873, 35360.19501617, 52126.63447019,
       27364.65512245, 55243.71815439, 41160.65740683, 46298.72370622,
       30068.62919602, 36953.90333158, 56993.15381371, 37189.51812084,
       55718.26510954, 35828.31590431,  7297.03304465, 40132.67129572,
       22820.06472206, 37222.99292251, 16858.05698898, 17553.86560423,
       28958.88851535, 26766.69956918, 40199.30191379, 27669.05628718,
       33390.83343215, 20737.18554581, 34155.84632661, 41436.37481897,
       16450.31712765, 52213.45426611, 32762.36885075, 50459.72063715,
      

In [27]:
r2 = r2_score(y_test, base_model.predict(x_test))
print(f"R-squared: {r2:.4f}")

R-squared: 0.9383


In [28]:
mean_squared_error_value = mean_squared_error(y_test, base_model.predict(x_test))
print(f"Mean Squared Error: {mean_squared_error_value:.4f}")

Mean Squared Error: 9478540.5091


In [29]:
mean_absolute_error_value = mean_absolute_error(y_test, base_model.predict(x_test))
print(f"Mean Absolute Error: {mean_absolute_error_value:.4f}")

Mean Absolute Error: 2408.5414
